# DisfluencyLab: Robust Speech Recognition on Disfluent Speech
## 21CSE453T — Speech Recognition, Activity Unit 3

**Research question:** How do pretrained Wav2Vec2-CTC and Whisper transcribe speech containing disfluencies, and how do reference-text normalization choices affect measured word error rate (WER)?

**Pipeline:** Speech → Feature Extraction → Log-Mel Spectrum → ASR → CTC → Text → TTS → Speech.

**Key idea:** every model is scored twice — against the *verbatim* reference (stutters kept) and against the *clean* reference (intended fluent text). The gap between the two shows whether a model transcribes or silently cleans up disfluency.

**Dataset:** UCLASS Release One recordings with corresponding orthographic transcripts, subject to the archive's usage and attribution conditions. The notebook keeps the original transcript text and creates a separate normalized version for evaluation. Dataset annotations are not assumed to be perfect ground truth without inspection.

**Reproducibility:** Start with a small set of matched WAV/transcript pairs. Keep filenames unchanged. Record model names, sample rate, and processing time. Results are generated by running the notebook; no scores are hardcoded.

> This notebook builds on a course activity reference notebook. New dataset handling and evaluation steps should be reviewed, understood, and cited appropriately in the final submission.

## 1. Setup

Run the installation cell, then run the dataset-loading cell. Before uploading, **compress your local `DisfluencyLab` folder into a ZIP file**. The loader accepts one ZIP containing the `data/audio` and `data/transcripts` folders (or the WAV and transcript files anywhere inside the ZIP).

Expected local layout:

```text
DisfluencyLab/
└── data/
    ├── audio/       # three .wav files
    └── transcripts/ # matching .orth or .txt files
```


In [1]:
# Google Colab dependencies
!pip -q install librosa soundfile matplotlib numpy pandas jiwer transformers accelerate gTTS graphviz


^C


In [ ]:
import os, re, json, math, subprocess, sys, time, html, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Image, Markdown

# Project and output folders
PROJECT_ROOT = Path('/content/DisfluencyLab')
DATA_DIR = PROJECT_ROOT / 'data'
AUDIO_DIR = DATA_DIR / 'audio'
TRANSCRIPT_DIR = DATA_DIR / 'transcripts'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
for folder in (AUDIO_DIR, TRANSCRIPT_DIR, VISUALS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Upload one ZIP of the local DisfluencyLab folder.
from google.colab import files
print("Upload ONE ZIP file containing your DisfluencyLab folder and its data/audio + data/transcripts folders.")
uploaded = files.upload()
zip_names = [name for name in uploaded if Path(name).suffix.lower() == '.zip']
if len(zip_names) != 1:
    raise ValueError("Please upload exactly one .zip file. On Windows, right-click the DisfluencyLab folder > Compress to ZIP file, then upload that ZIP.")

# Clear previous dataset files so old uploads cannot accidentally be mixed in.
for folder in (AUDIO_DIR, TRANSCRIPT_DIR):
    for old_file in folder.iterdir():
        if old_file.is_file():
            old_file.unlink()

# Extract safely to a temporary location, then collect matching file types recursively.
TEMP_EXTRACT_DIR = PROJECT_ROOT / '_uploaded_zip'
if TEMP_EXTRACT_DIR.exists():
    shutil.rmtree(TEMP_EXTRACT_DIR)
TEMP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
zip_path = Path('/content') / zip_names[0]
with zipfile.ZipFile(zip_path) as archive:
    for member in archive.infolist():
        destination = (TEMP_EXTRACT_DIR / member.filename).resolve()
        if not str(destination).startswith(str(TEMP_EXTRACT_DIR.resolve()) + os.sep) and destination != TEMP_EXTRACT_DIR.resolve():
            raise ValueError("The ZIP contains an unsafe file path. Please create a fresh ZIP from your project folder.")
    archive.extractall(TEMP_EXTRACT_DIR)

for source in TEMP_EXTRACT_DIR.rglob('*'):
    if not source.is_file():
        continue
    suffix = source.suffix.lower()
    if suffix == '.wav':
        shutil.copy2(source, AUDIO_DIR / source.name)
    elif suffix in {'.orth', '.txt'}:
        shutil.copy2(source, TRANSCRIPT_DIR / source.name)

# Match each WAV with a transcript sharing the same filename stem.
audio_files = sorted(AUDIO_DIR.glob('*.wav'))
transcript_files = {p.stem: p for p in list(TRANSCRIPT_DIR.glob('*.orth')) + list(TRANSCRIPT_DIR.glob('*.txt'))}
dataset_records = []
for audio_file in audio_files:
    transcript_file = transcript_files.get(audio_file.stem)
    if transcript_file is None:
        print(f"WARNING: no transcript matched {audio_file.name}")
        continue
    raw_text = transcript_file.read_text(encoding='utf-8', errors='replace').strip()
    dataset_records.append({
        'recording_id': audio_file.stem,
        'audio_path': str(audio_file),
        'transcript_path': str(transcript_file),
        'raw_reference': raw_text
    })

dataset = pd.DataFrame(dataset_records)
print(f"Found {len(audio_files)} WAV file(s) and {len(transcript_files)} transcript file(s) in the ZIP.")
print(f"Matched audio/transcript pairs: {len(dataset)} / {len(audio_files)} audio files")
if dataset.empty:
    raise ValueError("No matched pairs found. Check that each WAV and transcript have the same filename stem, e.g. M_0028_15y11m_1.wav and M_0028_15y11m_1.txt.")
display(dataset[['recording_id', 'audio_path', 'transcript_path']])

### Shared helpers
The normalisation, clean-reference, disfluency-counting, block-detection and WER functions live in `disfluency_utils.py`. The Streamlit app imports the **same file**, so notebook and app can never disagree. Copy it into your repo root after the run (it is also in the results ZIP).

In [ ]:
%%writefile /content/DisfluencyLab/disfluency_utils.py
"""Shared helpers for DisfluencyLab (used by the notebook AND app.py)."""
import html
import re

import numpy as np

FILLERS = {"um", "uh", "er", "erm", "ah", "hmm", "mm", "eh", "uhm", "umm"}


def clean_transcript_markup(text):
    """Conservative markup cleanup. Does not remove words."""
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_for_wer(text):
    """Lowercase + strip punctuation. Keeps fillers and repetitions (verbatim)."""
    text = clean_transcript_markup(text).lower()
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def collapse_repeats(words, max_n=3):
    """Collapse immediate repeats of 1..max_n word units: 'the the ball' -> 'the ball'."""
    changed = True
    while changed:
        changed = False
        for n in range(max_n, 0, -1):
            out, i = [], 0
            while i < len(words):
                if i + 2 * n <= len(words) and words[i:i + n] == words[i + n:i + 2 * n]:
                    out.extend(words[i:i + n])
                    i += 2 * n
                    changed = True
                else:
                    out.append(words[i])
                    i += 1
            words = out
    return words


def _strip_fragments(text):
    # stutter prefixes: "b-b-ball" -> "ball", "wa-wa-want" -> "want"
    text = re.sub(r"\b([a-z]{1,3})-(?:\1-)*(?=\1)", " ", text)
    # trailing broken word: "wa- " -> removed
    text = re.sub(r"\b[a-z']+-(?=\s|$)", " ", text)
    return text


def make_clean_reference(text):
    """Fluent 'intended text': drop fillers, fragments and immediate repetitions.
    Heuristic: legitimate repeats ('that that') are also collapsed. Inspect the output."""
    t = _strip_fragments(clean_transcript_markup(text).lower())
    words = [w for w in normalize_for_wer(t).split() if w not in FILLERS]
    return " ".join(collapse_repeats(words))


def count_disfluencies(text):
    """Rough counts of annotated disfluencies in a verbatim reference."""
    t = clean_transcript_markup(text).lower()
    words = normalize_for_wer(t).split()
    fillers = sum(w in FILLERS for w in words)
    fragments = len(re.findall(r"\b[a-z']+-(?=\s|$|[a-z])", t))
    content = [w for w in words if w not in FILLERS]
    repeats = len(content) - len(collapse_repeats(content))
    return {
        "words": len(words),
        "fillers": fillers,
        "fragments": fragments,
        "repeated_words": repeats,
        "disfluency_rate": round((fillers + fragments + repeats) / max(len(words), 1), 3),
    }


def detect_blocks(y, sr, min_gap=0.4, top_db=35):
    """Candidate blocks / long pauses: silent gaps >= min_gap s BETWEEN speech
    segments. Normal pauses also appear -> verify by listening."""
    import librosa
    intervals = librosa.effects.split(y, top_db=top_db, frame_length=1024, hop_length=256)
    gaps = []
    for (_, e0), (s1, _) in zip(intervals[:-1], intervals[1:]):
        if (s1 - e0) / sr >= min_gap:
            gaps.append((e0 / sr, s1 / sr))
    return gaps


def wer_breakdown(reference_norm, hypothesis_raw):
    """WER with S/D/I/H. `reference_norm` must already be normalised;
    the hypothesis is normalised here."""
    from jiwer import process_words
    hyp = normalize_for_wer(hypothesis_raw)
    if not reference_norm:
        return dict(WER=np.nan, S=0, D=0, I=0, H=0, N=0)
    if not hyp:  # jiwer dislikes empty hypotheses -> all deletions
        n = len(reference_norm.split())
        return dict(WER=1.0, S=0, D=n, I=0, H=0, N=n)
    o = process_words(reference_norm, hyp)
    return dict(WER=o.wer, S=o.substitutions, D=o.deletions, I=o.insertions,
                H=o.hits, N=o.hits + o.substitutions + o.deletions)


def alignment_html(reference_norm, hypothesis_raw):
    """Colour-coded word alignment (substitution / deletion / insertion)."""
    from jiwer import process_words
    hyp = normalize_for_wer(hypothesis_raw)
    if not reference_norm or not hyp:
        return "<i>Nothing to align.</i>"
    o = process_words(reference_norm, hyp)
    ref, hy = o.references[0], o.hypotheses[0]
    parts = []
    for c in o.alignments[0]:
        r, h = ref[c.ref_start_idx:c.ref_end_idx], hy[c.hyp_start_idx:c.hyp_end_idx]
        if c.type == "equal":
            parts.append(" ".join(r))
        elif c.type == "substitute":
            parts.append(f"<span style='background:rgba(255,170,0,.30);padding:0 3px;border-radius:3px'>"
                         f"<s>{' '.join(r)}</s>&rarr;{' '.join(h)}</span>")
        elif c.type == "delete":
            parts.append(f"<span style='background:rgba(255,70,70,.30);padding:0 3px;border-radius:3px'>"
                         f"<s>{' '.join(r)}</s></span>")
        elif c.type == "insert":
            parts.append(f"<span style='background:rgba(60,200,90,.30);padding:0 3px;border-radius:3px'>"
                         f"+{' '.join(h)}</span>")
    return " ".join(parts)

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from disfluency_utils import (normalize_for_wer, make_clean_reference, count_disfluencies,
                              detect_blocks, wer_breakdown, FILLERS)

RESULTS_DIR = PROJECT_ROOT / 'results'
TTS_DIR = RESULTS_DIR / 'tts'
for folder in (RESULTS_DIR, TTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000
N_FFT, HOP, N_MELS = 400, 160, 80      # 25 ms window / 10 ms hop / 80 mel bands (same as Whisper)

dataset['verbatim_ref'] = dataset['raw_reference'].map(normalize_for_wer)
dataset['clean_ref']    = dataset['raw_reference'].map(make_clean_reference)

counts = pd.DataFrame([count_disfluencies(t) for t in dataset['raw_reference']])
disfl_table = pd.concat([dataset[['recording_id']], counts], axis=1)
display(disfl_table)

# Audit the cleaning rule -- never trust it blindly.
for _, r in dataset.iterrows():
    print(f"\n[{r.recording_id}]\n  RAW     : {r.raw_reference[:300]}\n  VERBATIM: {r.verbatim_ref[:300]}\n  CLEAN   : {r.clean_ref[:300]}")

**Read the audit above.** `clean_ref` is produced by a heuristic (drop fillers like *um/uh*, drop fragments like *b-b-*, collapse immediate repeats). It also collapses legitimate repeats such as "that that". Fix the rule in `disfluency_utils.py` if your transcripts use different markup. Counts in the table are annotation-based, so they are only as good as the transcripts.

## 2. Audio inspection and waveform

For every recording we plot the waveform and mark **candidate blocks**: silent gaps of ≥ 0.4 s *between* speech segments, found with an energy threshold (`librosa.effects.split`). These are the "temporary disruptions in the normal vocal pattern" the activity asks about. Normal pauses are caught too, so confirm by listening.

In [ ]:
AUDIO, BLOCKS = {}, {}
for _, row in dataset.iterrows():
    rid = row['recording_id']
    info = sf.info(row['audio_path'])
    y = librosa.load(row['audio_path'], sr=TARGET_SR, mono=True)[0]   # resample to 16 kHz
    AUDIO[rid] = y
    BLOCKS[rid] = detect_blocks(y, TARGET_SR)
    print(f"{rid}: original {info.samplerate} Hz | {len(y)/TARGET_SR:.2f} s | {len(BLOCKS[rid])} candidate block(s)")

    fig, ax = plt.subplots(figsize=(12, 3.2))
    librosa.display.waveshow(y, sr=TARGET_SR, ax=ax, alpha=.8)
    for k, (s, e) in enumerate(BLOCKS[rid]):
        ax.axvspan(s, e, color='red', alpha=.25, label='candidate block' if k == 0 else None)
    ax.set(title=f'Waveform — {rid}', xlabel='Time (s)', ylabel='Amplitude')
    if BLOCKS[rid]: ax.legend(loc='upper right')
    fig.tight_layout()
    fig.savefig(VISUALS_DIR / f'{rid}_waveform.png', dpi=200, bbox_inches='tight')
    plt.show()
    display(Audio(y, rate=TARGET_SR))

## 3. Feature extraction and Log-Mel spectrum

Pipeline: **waveform → 25 ms frames (10 ms hop) → Hamming window → FFT → power spectrum → mel filterbank → log**. The first figure walks through one frame; the second shows the full result for every recording. A log-Mel spectrogram is a visualisation and model input, not a stutter detector by itself.

In [ ]:
# One frame, step by step (taken at the loudest point of the first recording)
rid0 = dataset.iloc[0]['recording_id']; y0 = AUDIO[rid0]
rms = librosa.feature.rms(y=y0, frame_length=N_FFT, hop_length=HOP)[0]
start = int(np.argmax(rms)) * HOP
frame = y0[start:start + N_FFT]
windowed = frame * np.hamming(N_FFT)
spectrum = np.abs(np.fft.rfft(windowed)) ** 2
fbank = librosa.filters.mel(sr=TARGET_SR, n_fft=N_FFT, n_mels=N_MELS)
mel_energy = fbank @ spectrum

fig, ax = plt.subplots(1, 5, figsize=(18, 3))
ax[0].plot(frame);                         ax[0].set_title('1. 25 ms frame')
ax[1].plot(windowed);                      ax[1].set_title('2. Hamming window')
ax[2].plot(10*np.log10(spectrum + 1e-10)); ax[2].set_title('3. Power spectrum (dB)')
ax[3].plot(fbank.T);                       ax[3].set_title('4. Mel filterbank (80)')
ax[4].plot(np.log(mel_energy + 1e-10));    ax[4].set_title('5. Log-Mel (one column)')
fig.tight_layout()
fig.savefig(VISUALS_DIR / 'feature_extraction_steps.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
def log_mel_of(sig):
    mel = librosa.feature.melspectrogram(y=sig, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP,
                                         n_mels=N_MELS, fmax=TARGET_SR // 2, power=2.0)
    return librosa.power_to_db(mel, ref=np.max)

for rid, y in AUDIO.items():
    stft_db = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)), ref=np.max)
    lm = log_mel_of(y)
    fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    librosa.display.waveshow(y, sr=TARGET_SR, ax=ax[0]); ax[0].set_title(f'{rid} — waveform')
    im1 = librosa.display.specshow(stft_db, sr=TARGET_SR, hop_length=HOP, x_axis='time', y_axis='hz', ax=ax[1])
    ax[1].set_title('Linear-frequency spectrogram (STFT, dB)'); fig.colorbar(im1, ax=ax[1], format='%+2.0f dB')
    im2 = librosa.display.specshow(lm, sr=TARGET_SR, hop_length=HOP, x_axis='time', y_axis='mel', ax=ax[2])
    ax[2].set_title(f'Log-Mel spectrogram ({N_MELS} bands)'); fig.colorbar(im2, ax=ax[2], format='%+2.0f dB')
    for a in ax:
        for s, e in BLOCKS[rid]:
            a.axvspan(s, e, color='red', alpha=.15)
    fig.tight_layout()
    fig.savefig(VISUALS_DIR / f'{rid}_log_mel.png', dpi=200, bbox_inches='tight')
    plt.show()

**What to look for:** repetitions show up as repeated, similarly shaped bursts; prolongations as long horizontal bands with steady harmonics; blocks as dark vertical gaps (red). Describe what you see in *your* recordings here before moving on.

## 4. ASR architecture diagrams
The overall flow used for this activity is: **Speech → Feature Extraction → Log-Mel Spectrum → ASR → CTC → Text → TTS → Speech**. After it, the internals of the two ASR families being compared.

In [ ]:
from graphviz import Digraph

dot = Digraph(format='png')
dot.attr(rankdir='LR')
for node, label in [
    ('A','Speech audio\n(stutter / hiccup)'),
    ('B','Feature Extraction\nframing + windowing'),
    ('C','Log-Mel Spectrum\n80 Mel bands'),
    ('D','ASR Encoder\nacoustic features'),
    ('E','CTC\nblank + alignment'),
    ('F','Decoding\ncollapse repeats'),
    ('G','Text transcript'),
    ('H','TTS\ntext-to-speech'),
    ('I','Synthesized speech')]:
    dot.node(node, label, shape='box', style='rounded')
for u,v in zip('ABCDEFGH','BCDEFGHI'):
    dot.edge(u,v)

# Render architecture diagram to visuals/architecture.png
arch_output = VISUALS_DIR / 'architecture'
rendered_file = dot.render(str(arch_output), cleanup=True)
print('Architecture diagram saved to:', rendered_file)
display(Image(filename=rendered_file))


In [ ]:
from graphviz import Digraph

def arch(title, nodes, color):
    g = Digraph(format='png'); g.attr(rankdir='LR', label=title, labelloc='t', fontsize='16')
    for k, (nid, label) in enumerate(nodes):
        g.node(nid, label, shape='box', style='rounded,filled', fillcolor=color)
        if k: g.edge(nodes[k-1][0], nid)
    return g

w2v = arch('Wav2Vec2-base-960h (encoder-only, CTC)', [
    ('a','Raw waveform\n16 kHz'), ('b','CNN feature encoder\n7 conv layers, 20 ms stride'),
    ('c','Transformer\ncontext network\n12 layers'), ('d','Linear layer\n32 characters + blank'),
    ('e','CTC decoding\nargmax -> collapse\n-> remove blank'), ('f','Text')], '#dbeafe')
whi = arch('Whisper (encoder-decoder, autoregressive)', [
    ('a','Waveform\n30 s window'), ('b','80-bin log-Mel'), ('c','Conv stem\n(2 conv + GELU)'),
    ('d','Transformer encoder'), ('e','Transformer decoder\ncross-attention'),
    ('f','Token-by-token\ndecoding\n(beam/greedy)'), ('g','Text')], '#dcfce7')
tts = arch('TTS (gTTS / typical neural pipeline)', [
    ('a','Text'), ('b','Text normalisation\n+ phonemes'), ('c','Acoustic model\ntext -> mel'),
    ('d','Vocoder\nmel -> waveform'), ('e','Fluent speech')], '#fef9c3')
for name, g in (('wav2vec2_arch', w2v), ('whisper_arch', whi), ('tts_arch', tts)):
    display(Image(filename=g.render(str(VISUALS_DIR / name), cleanup=True)))

**Key contrast:** Wav2Vec2 outputs one label per 20 ms frame, independently, so repeated sounds tend to be *kept* as fragments. Whisper's decoder is a language model conditioned on the audio, so it tends to *rewrite* disfluent speech into fluent text. The results below test this.

## 5. CTC explanation
**Connectionist Temporal Classification (CTC)** trains a model when the input (frames) and target (characters) have different lengths and no known alignment.

* The model outputs, for each frame $t$, a probability distribution over the vocabulary **plus a blank** symbol $\varnothing$.
* A frame-level **path** $\pi$ is mapped to text by $\mathcal{B}$: **merge repeats, then delete blanks**.
* The probability of a target $y$ sums over *all* paths that collapse to it, and training minimises the negative log of that sum:

$$P(y\mid x)=\sum_{\pi:\ \mathcal{B}(\pi)=y}\ \prod_{t=1}^{T} p_t(\pi_t\mid x),\qquad \mathcal{L}_{CTC}=-\log P(y\mid x)$$

* The **blank** is what lets genuine double letters survive ("ball": `b a l _ l`), because blanks stop repeats being merged.
* **Stutter link:** a stuttered "b-b-ball" is acoustically three bursts. CTC has no notion of intent, so it is free to emit `b b ball`, `bball` or `ball` depending on how confident each frame is. The real posteriorgram in Section 6b shows what happened on your audio.

In [ ]:
# Small CTC-style decoding demonstration (toy sequence)
path = ['_', 'I', 'I', '_', 'W', 'W', 'A', 'N', 'T', '_']
collapsed = []
prev = None
for token in path:
    if token != prev:
        collapsed.append(token)
    prev = token
ctc_text = ''.join(t for t in collapsed if t != '_')
print('Raw CTC path     :', path)
print('After collapsing :', collapsed)
print('After removing _ :', ctc_text)


In [ ]:
# Why the blank matters: "ball" needs a blank between the two l's
def ctc_collapse(path, blank='_'):
    merged = [t for i, t in enumerate(path) if i == 0 or t != path[i-1]]
    return ''.join(t for t in merged if t != blank)

print(ctc_collapse(list('b_a_l_l_')), ' <- with blank between l l  (double letter kept)')
print(ctc_collapse(list('b_a_ll__')), ' <- without blank (the l l merge)')
print(ctc_collapse(list('bb_b_ba_a_lll_')), ' <- stutter-like path: separate bursts split by blanks are all kept')

## 6. ASR models and transcription

Four systems on the **same matched recordings**, with a warm-up pass so timings exclude one-off start-up cost:

| System | Family | Notes |
|---|---|---|
| Wav2Vec2-base-960h | CTC, encoder-only | trained on fluent read speech |
| Whisper-tiny.en | encoder-decoder | smallest Whisper |
| Whisper-base.en | encoder-decoder | larger, same family |
| Whisper-base.en + verbatim prompt | encoder-decoder | prompt contains *um / uh / repeats* to nudge it to keep disfluencies (dropped automatically if the installed `transformers` rejects prompts) |

`Real-time factor (RTF) = inference seconds ÷ audio seconds`; below 1.0 is faster than real time.

In [ ]:
from transformers import AutoProcessor, AutoModelForCTC, pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
dev_str = 'cuda' if device == 0 else 'cpu'
print('Inference device:', dev_str)

ctc_processor = AutoProcessor.from_pretrained('facebook/wav2vec2-base-960h')
ctc_model = AutoModelForCTC.from_pretrained('facebook/wav2vec2-base-960h').to(dev_str).eval()

whisper_pipes = {
    'Whisper-tiny.en': pipeline('automatic-speech-recognition', model='openai/whisper-tiny.en', chunk_length_s=30, device=device),
    'Whisper-base.en': pipeline('automatic-speech-recognition', model='openai/whisper-base.en', chunk_length_s=30, device=device),
}
VERBATIM_PROMPT = "Um, I-I-I uh want, uh, the the ball."

CTC_STORE = {}   # recording_id -> frame posteriors, used in section 6b

def run_wav2vec2(sig):
    inputs = ctc_processor(sig, sampling_rate=TARGET_SR, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = ctc_model(inputs.input_values.to(dev_str)).logits
    probs = logits.softmax(-1)[0].cpu().numpy()
    text = ctc_processor.batch_decode(logits.argmax(-1).cpu())[0].lower().strip()
    return text, probs

def run_whisper(pipe, sig, prompt=None):
    kw = {}
    if prompt:
        ids = pipe.tokenizer.get_prompt_ids(prompt, return_tensors='pt').to(pipe.model.device)
        kw['generate_kwargs'] = {'prompt_ids': ids}
    text = pipe({'raw': sig, 'sampling_rate': TARGET_SR}, **kw)['text'].lower().strip()
    if prompt and text.startswith(prompt.lower()):           # some versions echo the prompt
        text = text[len(prompt):].strip()
    return text

TRANSCRIBERS = {
    'Wav2Vec2-base-960h':  lambda s: run_wav2vec2(s)[0],
    'Whisper-tiny.en':     lambda s: run_whisper(whisper_pipes['Whisper-tiny.en'], s),
    'Whisper-base.en':     lambda s: run_whisper(whisper_pipes['Whisper-base.en'], s),
    'Whisper-base.en + verbatim prompt':
                           lambda s: run_whisper(whisper_pipes['Whisper-base.en'], s, VERBATIM_PROMPT),
}
n_params = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
MODEL_PARAMS = {
    'Wav2Vec2-base-960h': n_params(ctc_model),
    'Whisper-tiny.en': n_params(whisper_pipes['Whisper-tiny.en'].model),
    'Whisper-base.en': n_params(whisper_pipes['Whisper-base.en'].model),
    'Whisper-base.en + verbatim prompt': n_params(whisper_pipes['Whisper-base.en'].model),
}

# Warm-up (also checks the prompt variant works on this install)
silence = np.zeros(TARGET_SR, dtype=np.float32)
for name in list(TRANSCRIBERS):
    try:
        TRANSCRIBERS[name](silence)
    except Exception as err:
        print(f'Dropping "{name}": {type(err).__name__}: {str(err)[:120]}')
        TRANSCRIBERS.pop(name)

rows = []
for rid, sig in AUDIO.items():
    dur = len(sig) / TARGET_SR
    _, CTC_STORE[rid] = run_wav2vec2(sig)
    for name, fn in TRANSCRIBERS.items():
        t0 = time.perf_counter(); text = fn(sig); sec = time.perf_counter() - t0
        rows.append({'recording_id': rid, 'model': name, 'prediction': text,
                     'inference_seconds': round(sec, 3), 'audio_seconds': round(dur, 2),
                     'rtf': round(sec / dur, 3)})
    print('Completed', rid)

asr_results = pd.DataFrame(rows)
display(asr_results)

### 6b. Real CTC output on your audio
Below: the **posteriorgram** (probability of every character at every 20 ms frame), the raw frame-level path with `_` = blank and `|` = word boundary, and the collapsed text. Choose `WINDOW_START` to zoom in on a stutter or block from the waveform plot.

In [ ]:
rid = dataset.iloc[0]['recording_id']
WINDOW_START, WINDOW_LEN = 0.0, 4.0          # seconds -- change to inspect a stutter region
STRIDE = 0.02                                 # wav2vec2: one frame per 20 ms

probs = CTC_STORE[rid]
vocab = ctc_processor.tokenizer.convert_ids_to_tokens(list(range(probs.shape[1])))
blank_id = ctc_processor.tokenizer.pad_token_id
a, b = int(WINDOW_START / STRIDE), int((WINDOW_START + WINDOW_LEN) / STRIDE)
ids = probs.argmax(-1)

label = lambda i: '_' if i == blank_id else vocab[i]
raw_path = [label(i) for i in ids[a:b]]
print('Raw frame path :', ''.join(raw_path))
print('Collapsed text :', ctc_collapse([t if t != '|' else ' ' for t in raw_path]).replace('  ', ' '))
print(f'Blank frames   : {np.mean(ids == blank_id):.0%} of all frames (CTC is mostly blank)')

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(probs[a:b].T, aspect='auto', origin='lower', cmap='viridis',
               extent=[WINDOW_START, WINDOW_START + (b - a) * STRIDE, -.5, len(vocab) - .5])
ax.set_yticks(range(len(vocab))); ax.set_yticklabels([vocab[i] for i in range(len(vocab))], fontsize=7)
for s, e in BLOCKS[rid]:
    ax.axvspan(s, e, color='red', alpha=.2)
ax.set(title=f'CTC posteriorgram — {rid}', xlabel='Time (s)', ylabel='Token')
fig.colorbar(im, ax=ax, label='P(token | frame)')
fig.tight_layout(); fig.savefig(VISUALS_DIR / f'{rid}_ctc_posteriorgram.png', dpi=200, bbox_inches='tight')
plt.show()

**Interpretation to write:** in the window, how many consecutive frames carry the same letter? Where a stutter repeats a sound, does the path contain separate bursts divided by blanks (kept as a repeat) or one merged run (silently lost)? This is the mechanism behind Wav2Vec2's behaviour on disfluent speech.

## 7. Reference transcripts and WER (verbatim vs clean)

$$WER=\frac{S+D+I}{N}$$

* **Verbatim WER** — reference keeps fillers, fragments and repetitions. A model that cleans speech up is *penalised* with deletions.
* **Clean WER** — reference is the fluent intended text. A model that transcribes every stutter is *penalised* with insertions.
* The same model can look good on one and bad on the other, so **both are reported**. Hypotheses are only lower-cased and stripped of punctuation in both cases.
* Pooled WER (total errors ÷ total reference words) is reported alongside the per-recording mean, because short recordings otherwise get equal weight.

In [ ]:
from jiwer import process_words, visualize_alignment

wer_rows = []
for _, r in asr_results.iterrows():
    ref_row = dataset.loc[dataset.recording_id == r.recording_id].iloc[0]
    for ref_type, ref in (('verbatim', ref_row.verbatim_ref), ('clean', ref_row.clean_ref)):
        res = wer_breakdown(ref, r.prediction)
        wer_rows.append({'recording_id': r.recording_id, 'model': r.model, 'reference_type': ref_type,
                         'WER': res['WER'], 'substitutions': res['S'], 'deletions': res['D'],
                         'insertions': res['I'], 'hits': res['H'], 'reference_words': res['N'],
                         'inference_seconds': r.inference_seconds, 'rtf': r.rtf})
wer_table = pd.DataFrame(wer_rows)
display(wer_table.style.format({'WER': '{:.3f}', 'inference_seconds': '{:.2f}', 'rtf': '{:.3f}'}))
print('WER can exceed 1.0 when there are many insertions.')

# Word-level alignment for the first recording (verbatim reference)
first = dataset.iloc[0]; fid = first.recording_id
for name in TRANSCRIBERS:
    hyp = normalize_for_wer(asr_results.query('recording_id == @fid and model == @name').prediction.iloc[0])
    if hyp:
        print(f'\n=== {name} on {first.recording_id} (verbatim) ===')
        print(visualize_alignment(process_words(first.verbatim_ref, hyp)))

### Which errors come from which disfluency?
Compare **deletions** against the annotated disfluency counts: if a model's deletions ≈ the number of fillers + repeats, it is removing disfluencies rather than mishearing words. If its **insertions** against the clean reference are high, it is transcribing the stutter.

In [ ]:
v = wer_table[wer_table.reference_type == 'verbatim'].merge(disfl_table, on='recording_id')
v['annotated_disfluent_tokens'] = v.fillers + v.repeated_words + v.fragments
display(v[['recording_id', 'model', 'deletions', 'annotated_disfluent_tokens', 'insertions', 'substitutions']]
        .sort_values(['recording_id', 'model']).reset_index(drop=True))

## 8. ASR comparison table

Do not generalise from this small sample to all speakers or types of disfluency.

In [ ]:
wer_table['errors'] = wer_table.substitutions + wer_table.deletions + wer_table.insertions
g = (wer_table.groupby(['model', 'reference_type'])
     .agg(recordings=('recording_id', 'nunique'), mean_WER=('WER', 'mean'), errors=('errors', 'sum'),
          reference_words=('reference_words', 'sum'), S=('substitutions', 'sum'),
          D=('deletions', 'sum'), I=('insertions', 'sum')))
g['pooled_WER'] = g.errors / g.reference_words
g = g.reset_index()
v_, c_ = (g[g.reference_type == k].set_index('model') for k in ('verbatim', 'clean'))
speed = asr_results.groupby('model').agg(mean_inference_seconds=('inference_seconds', 'mean'),
                                         mean_RTF=('rtf', 'mean'))
comparison = pd.DataFrame({
    'verbatim_pooled_WER': v_.pooled_WER, 'clean_pooled_WER': c_.pooled_WER,
    'verbatim_mean_WER': v_.mean_WER, 'clean_mean_WER': c_.mean_WER,
    'verbatim_S': v_.S, 'verbatim_D': v_.D, 'verbatim_I': v_.I,
    'clean_S': c_.S, 'clean_D': c_.D, 'clean_I': c_.I,
}).join(speed)
comparison['params_M'] = pd.Series(MODEL_PARAMS)
comparison = comparison.reset_index().sort_values('verbatim_pooled_WER')
display(comparison.style.format({c: '{:.3f}' for c in comparison.columns
                                 if comparison[c].dtype.kind == 'f'}))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
comparison.set_index('model')[['verbatim_pooled_WER', 'clean_pooled_WER']].plot.bar(ax=ax[0])
ax[0].set(title='Pooled WER: verbatim vs clean reference', ylabel='WER (lower is better)', xlabel='')
ax[0].tick_params(axis='x', rotation=20)
sdi = wer_table[wer_table.reference_type == 'verbatim'].groupby('model')[['substitutions', 'deletions', 'insertions']].sum()
sdi.plot.bar(stacked=True, ax=ax[1], color=['#f59e0b', '#ef4444', '#22c55e'])
ax[1].set(title='Error types vs verbatim reference', ylabel='Word errors', xlabel='')
ax[1].tick_params(axis='x', rotation=20)
fig.tight_layout(); fig.savefig(VISUALS_DIR / 'model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

### Auto-generated findings
These sentences are computed from the tables above, so they stay correct if you change the data. Add your own interpretation underneath.

In [ ]:
best_v = comparison.sort_values('verbatim_pooled_WER').iloc[0]
best_c = comparison.sort_values('clean_pooled_WER').iloc[0]
print(f"Best on verbatim reference : {best_v.model} (pooled WER {best_v.verbatim_pooled_WER:.3f})")
print(f"Best on clean reference    : {best_c.model} (pooled WER {best_c.clean_pooled_WER:.3f})")
for _, r in comparison.iterrows():
    style = ('removes disfluencies (deletions dominate vs verbatim)' if r.verbatim_D > r.verbatim_I
             else 'keeps/creates extra words (insertions dominate vs verbatim)')
    gap = r.verbatim_pooled_WER - r.clean_pooled_WER
    print(f"- {r.model}: verbatim-clean WER gap = {gap:+.3f}; {style}; RTF {r.mean_RTF:.2f}; {r.params_M:.0f}M params")
print("\nCaveat: only", dataset.shape[0], "recordings -- indicative, not statistical evidence.")

## 9. TTS: text back to speech

TTS stage of the pipeline: **text → text normalisation / phonemes → acoustic model (text → mel) → vocoder (mel → waveform)**. gTTS is a cloud service (internet required). We synthesise the transcript of the lowest-clean-WER model for **every** recording, then compare the synthetic audio with the original:

* the TTS output is *fluent by construction*, so the spectrogram shows the disfluency has been normalised away;
* a **round-trip check** (TTS audio → Whisper → text) shows how intelligible the synthetic speech is.

In [ ]:
from gtts import gTTS

TTS_MODEL = comparison.sort_values('clean_pooled_WER').iloc[0]['model']
print('Using transcripts from:', TTS_MODEL)
tts_rows = []
for rid in AUDIO:
    text = asr_results.query('recording_id == @rid and model == @TTS_MODEL').prediction.iloc[0]
    if not text.strip():
        print(f'Skipping {rid}: empty transcript'); continue
    mp3 = TTS_DIR / f'{rid}.mp3'; wav = TTS_DIR / f'{rid}.wav'
    gTTS(text=text, lang='en').save(str(mp3))
    sig = librosa.load(mp3, sr=TARGET_SR, mono=True)[0]
    sf.write(wav, sig, TARGET_SR); mp3.unlink()

    rt_text = TRANSCRIBERS.get('Whisper-base.en', next(iter(TRANSCRIBERS.values())))(sig)
    rt = wer_breakdown(normalize_for_wer(text), rt_text)
    tts_rows.append({'recording_id': rid, 'tts_source_model': TTS_MODEL, 'tts_text': text,
                     'original_seconds': round(len(AUDIO[rid]) / TARGET_SR, 2),
                     'tts_seconds': round(len(sig) / TARGET_SR, 2),
                     'original_blocks': len(BLOCKS[rid]),
                     'tts_blocks': len(detect_blocks(sig, TARGET_SR)),
                     'round_trip_WER': round(rt['WER'], 3)})
tts_table = pd.DataFrame(tts_rows)
display(tts_table.drop(columns='tts_text'))

rid = tts_table.iloc[0].recording_id
tts_sig = librosa.load(TTS_DIR / f'{rid}.wav', sr=TARGET_SR)[0]
fig, ax = plt.subplots(2, 2, figsize=(14, 6))
for col, (title, sig) in enumerate((('Original (disfluent)', AUDIO[rid]), ('TTS output (fluent)', tts_sig))):
    librosa.display.waveshow(sig, sr=TARGET_SR, ax=ax[0, col]); ax[0, col].set_title(title)
    im = librosa.display.specshow(log_mel_of(sig), sr=TARGET_SR, hop_length=HOP, x_axis='time', y_axis='mel', ax=ax[1, col])
fig.tight_layout(); fig.savefig(VISUALS_DIR / f'{rid}_original_vs_tts.png', dpi=200, bbox_inches='tight')
plt.show()
print('Original :'); display(Audio(AUDIO[rid], rate=TARGET_SR))
print('TTS      :'); display(Audio(tts_sig, rate=TARGET_SR))

**Interpretation to write:** compare duration, pause structure and block count. TTS is evenly paced with no repetitions, confirming the full pipeline converts disfluent speech to fluent speech. Note that gTTS is not a perceptual quality metric.

## 10. Export for the Streamlit app

Writes everything the app reads into `results/` and downloads a ZIP. Unzip it into your project root (the ZIP also contains `disfluency_utils.py`).

In [ ]:
asr_results.to_csv(RESULTS_DIR / 'asr_results.csv', index=False)
wer_table.drop(columns='errors').to_csv(RESULTS_DIR / 'wer_results.csv', index=False)
comparison.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
disfl_table.to_csv(RESULTS_DIR / 'disfluency_counts.csv', index=False)
tts_table.to_csv(RESULTS_DIR / 'tts_results.csv', index=False)
for rid, p in CTC_STORE.items():
    np.savez_compressed(RESULTS_DIR / f'ctc_{rid}.npz', probs=p.astype(np.float16),
                        vocab=np.array(ctc_processor.tokenizer.convert_ids_to_tokens(list(range(p.shape[1])))),
                        blank_id=blank_id, stride=STRIDE)

shutil.copy(PROJECT_ROOT / 'disfluency_utils.py', RESULTS_DIR / 'disfluency_utils.py')
archive = shutil.make_archive('/content/disfluencylab_results', 'zip', root_dir=PROJECT_ROOT,
                              base_dir='results')
print('Exported:', sorted(p.name for p in RESULTS_DIR.iterdir()))
files.download(archive)

### Submission and dataset notes

- Cite the UCLASS dataset and follow its terms for research/teaching use; do not publicly redistribute recordings (or this repo's `data/` folder) unless permitted. The speakers are minors: keep the data private.
- Preserve original transcript files. The clean reference is a heuristic; document any change you make to it.
- Three recordings validate the pipeline; they do not support statistical conclusions, and no model should be called "best" in general.
- Pretrained models are used as-is; nothing is trained from scratch.
- Credit the original course reference notebook this builds on.
- Timings depend on hardware (state whether you used CPU or GPU).